# Model Finetuning

The following notebook will finetune the model by training it on the sft datasets appositely created.

### Imports

In [ ]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from datasets import load_from_disk
import re
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer




### Path definitions

In [ ]:
PROJECT_ROOT = Path("/mnt/data/projects/Multilingual_finetune").resolve()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SFT = DATA_PROCESSED / "sft"

ARTIFACTS_ROOT = Path("/mnt/data/project_artifacts")
MODEL_CACHE = ARTIFACTS_ROOT / "models_cache"
CHECKPOINT_DIR = ARTIFACTS_ROOT / "checkpoints"
LOGS_DIR = ARTIFACTS_ROOT / "logs"

for p in [DATA_SFT, MODEL_CACHE, CHECKPOINT_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("DATA_SFT:", DATA_SFT)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)
print("MODEL_CACHE:", MODEL_CACHE)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

train_sft_path = DATA_SFT / "train_sft"
validation_sft_path = DATA_SFT / "validation_sft"


PROJECT_ROOT: /home/ubuntu/Multilingual-Finetuning-Mistral-3B
DATA_PROCESSED: /home/ubuntu/Multilingual-Finetuning-Mistral-3B/data/processed
DATA_SFT: /home/ubuntu/Multilingual-Finetuning-Mistral-3B/data/processed/sft
ARTIFACTS_ROOT: /mnt/data/project_artifacts


### Loading the datasets
After loading we run a quick sanity check.

In [5]:
train_sft = load_from_disk(train_sft_path)
validation_sft = load_from_disk(validation_sft_path)
print("Loaded existing SFT datasets.")

sample = train_sft[0]
for key, value in sample.items():
    print(key, ":", value)


Loaded existing SFT datasets.
language : English
language_code : eng
messages : [{'content': 'What initiatives are in place to promote sustainable tourism in Sri Lanka, balancing economic benefits with environmental conservation?', 'role': 'user'}, {'content': "Sri Lanka has been actively promoting sustainable tourism by encouraging eco-friendly practices, community engagement, and wildlife conservation. Certification programs for eco-friendly accommodations and responsible tourism campaigns aim to ensure that tourism contributes positively to both the economy and the environment. Here are some key examples:\n\n1. Policy and Regulatory Framework:\n\nNational Sustainable Tourism Policy 2018: This policy outlines a comprehensive framework for sustainable tourism development, including responsible visitor management, environmental protection, and community engagement.\nNational Ecotourism Strategy: This strategy focuses on promoting eco-friendly tourism practices and experiences, such as 

### Model loading
The next cell loads the, already downloaded, model that is going to be finetuned from the cache.

##### Padding

Padding tokens are special tokens added to sequences so that all inputs in a batch have the same length. This is necessary because models process data in fixed-size tensors.

The padding tokens are ignored by the model thanks to the mask used by the attention.

In [6]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=str(MODEL_CACHE),
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(type(tokenizer))
print("Pad token:", tokenizer.pad_token)


<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
Pad token: <|endoftext|>


In [7]:
if torch.cuda.is_available():
    # GPU path
    chosen_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=str(MODEL_CACHE),
        torch_dtype=chosen_dtype,
        device_map="auto",
    )
else:
    # CPU path
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        cache_dir=str(MODEL_CACHE),
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True,
    )

model.eval()

print("Model loaded.")
print(type(model))
print(type(tokenizer))
print("CUDA available:", torch.cuda.is_available())
print("First parameter dtype:", next(model.parameters()).dtype)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 434/434 [00:49<00:00,  8.77it/s]


Model loaded.
<class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>
CUDA available: False
First parameter dtype: torch.float32


### Training setup

We begin by defining the LoRA fine tuning setup, which consist in defining the low-rank matrices that are being trained.

This is the key idea: instead of training and reweighting the whole set of parameters we are only tuning a subset of the parameters.

This approach is more efficient and is cheaper in memory, storage and training time.

In [9]:
peft_config = LoraConfig(
    r=16,   # is the rank of the adapters, higher means more trianable capacity but more memory usage
    lora_alpha=32,  # scaling factor for the updates
    lora_dropout=0.05,  # dropout applied during training for the bit regularization
    bias="none",    # not training bias terms
    task_type=TaskType.CAUSAL_LM,  
    target_modules=[    # These will be the model sublayers where the LoRA adapters will be inserted (attention and MLP projection layers)
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


#### Training Configuration

The following cell defines all the details setups and parameters that will be used during the training.

In [ ]:
run_name = "qwen25_3b_multilingual_lora_run1"

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR / run_name),  # Where to save the outputs

    per_device_train_batch_size=1,  # how big the batches are
    per_device_eval_batch_size=1,   
    gradient_accumulation_steps=8,  # how often to evaluate and save

    learning_rate=2e-4, # the step size of the training optimization
    num_train_epochs=1, # per each epoch we have a full pass over the training set

    logging_steps=10,   # print logs each 10 steps
    eval_strategy="steps",
    eval_steps=50,  # run validation each 50 steps
    save_strategy="steps",
    save_steps=50,  # save checkpoints each 50 steps
    save_total_limit=2, # keep only the two most recent checkpoints so that storage does not explode

    load_best_model_at_end=True,    # After training reload the checkpoint with the best validation loss
    metric_for_best_model="eval_loss",  # The best checkpoint is chosen as a function of the validation loss
    greater_is_better=False,

    report_to="none",

    max_length=512,    # maximum tokenized sequence length used in training
    packing=False,     # do not concatenate multiple short samples into one sequence

    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),  # use mixed precision on GPU if available otherwise stay in full precision
)


##### Control Runs
Define some control flags to do some sanity check runs on subsets of the whole dataset.

In [ ]:
PILOT_MODE = True
RUN_TRAINING = False   # keep False on CPU, switch to True when you are ready on GPU

train_sft_small = train_sft.shuffle(seed=42).select(
    range(min(300, len(train_sft)))
)

validation_sft_small = validation_sft.shuffle(seed=42).select(
    range(min(60, len(validation_sft)))
)

print(len(train_sft_small), len(validation_sft_small))


In [ ]:
if PILOT_MODE:
    train_run_ds = train_sft_small
    validation_run_ds = validation_sft_small
else:
    train_run_ds = train_sft
    validation_run_ds = validation_sft

print("Train run size:", len(train_run_ds))
print("Validation run size:", len(validation_run_ds))


### Training
The following cells are the ones where the actual training takes part.

In [ ]:
trainer = SFTTrainer(
    model=model,        # base model to finetune
    args=training_args,     # Training settings defined before
    train_dataset=train_run_ds, # dataset used for training
    eval_dataset=validation_run_ds, # dataset used for validation
    processing_class=tokenizer, # the tokenizer used to process the text
    peft_config=peft_config,    # the LoRA configurations so that training will use adapters instead of doing a full fine-tuning
)

print(trainer)
print("Output dir:", training_args.output_dir)
print(trainer.model)


In [ ]:
if RUN_TRAINING and not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Notebook setup is fine, but do not launch 3B fine-tuning on CPU.")

if RUN_TRAINING:
    trainer.train()     # ACTUAL TRAINING START COMMAND
else:
    print("RUN_TRAINING is False, so training is skipped.")

if RUN_TRAINING:
    trainer.save_model()    # saves the trained model
    tokenizer.save_pretrained(training_args.output_dir) # saves the tokenizer
    print("Saved model/tokenizer to:", training_args.output_dir)


### Training sucess testing
We run a quick manual inerence test after the finetuning training.

In [ ]:
messages = [sample["messages"][0]]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(prompt_text)


inputs = tokenizer(prompt_text, return_tensors="pt")

with torch.no_grad():
    outputs = trainer.model.generate(   # If the training was successfull then trainer.model now contains the LoRA adapted trained version. Note this works only within the same notebook session where you do not need to reload the finetuned model, because the finetuned model object is already inside the trainer.
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

prompt_len = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][prompt_len:]

prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("PREDICTION:\n", prediction)
print("\nTARGET:\n", sample["messages"][1]["content"])
